# Capstone — Lane 3: Structured Content Archetype Clustering
**FlyRank AI Corp — Search Intelligence Internship**

**Goal:** group content into clear, evidence-backed performance archetypes
(e.g. *Star / Protect*, *Sleeper / Improve*, *Declining / Rewrite*,
*Redundant / Merge*, *Dead Weight / Prune*, *Stable / Monitor*) and map each
archetype to a recommended action with a reason code.

This notebook is self-contained: **Step 0 → Step 9**. Run top to bottom.
Public rule respected throughout — no client names, domains, URLs, private
queries, credentials, or raw exports appear in any output cell.


## Step 0 — Setup: imports, HF token, config

In [ ]:
# Step 0.1 — imports
import os
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", 50)
np.random.seed(42)


In [ ]:
# Step 0.2 — Hugging Face read token
# Set this as an environment variable BEFORE launching Jupyter, e.g. in your shell:
#   export HF_TOKEN="hf_xxx..."
# Never hardcode the token in the notebook — it must not appear in git history.
HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN not set. Export it in your shell before starting Jupyter."

con = duckdb.connect()
con.execute("SET hf_token = ?", [HF_TOKEN]) if False else None  # duckdb picks up HF creds via httpfs config below
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"SET s3_endpoint='huggingface.co';") if False else None
os.environ["HF_TOKEN"] = HF_TOKEN  # duckdb's hf:// reader uses this env var directly


In [ ]:
# Step 0.3 — config: dataset path, month window, output paths
HF_DATASET_URI = "hf://datasets/<flyrank-dataset-org>/<dataset-name>/fact_content_daily_performance/*.parquet"
MONTH_WINDOW = "2026-03"   # match your ML-04 dev slice; widen once the pipeline is validated

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)


## Step 1 — Load data (DuckDB over hf://)
Aggregate at query time so nothing raw (per-query, per-URL detail) ever
lands in a notebook cell output — only page-level, public-safe aggregates.


In [ ]:
# Step 1.1 — pull page-level daily aggregates for the month window
query = f"""
SELECT
    page_id,                         -- use an internal/public-safe id, never the raw URL
    date,
    impressions,
    clicks,
    avg_position,
    ctr,
    content_age_days,
    word_count
FROM read_parquet('{HF_DATASET_URI}')
WHERE strftime(date, '%Y-%m') = '{MONTH_WINDOW}'
"""

df_raw = con.execute(query).df()
print(df_raw.shape)
df_raw.head()


## Step 2 — Feature engineering: one row per page

In [ ]:
# Step 2.1 — aggregate daily rows into page-level features
# Trend features use a simple first-half vs second-half split of the month
# (time-aware, avoids peeking at the full-month average as if it were a snapshot).

def half_split_trend(g, col):
    g = g.sort_values("date")
    mid = len(g) // 2
    first, second = g[col].iloc[:mid], g[col].iloc[mid:]
    if len(first) == 0 or first.mean() == 0:
        return 0.0
    return (second.mean() - first.mean()) / (abs(first.mean()) + 1e-6)

rows = []
for pid, g in df_raw.groupby("page_id"):
    rows.append({
        "page_id": pid,
        "avg_position_mean": g["avg_position"].mean(),
        "ctr_mean": g["ctr"].mean(),
        "impressions_sum": g["impressions"].sum(),
        "clicks_sum": g["clicks"].sum(),
        "impressions_trend": half_split_trend(g, "impressions"),
        "ctr_trend": half_split_trend(g, "ctr"),
        "position_trend": half_split_trend(g, "avg_position"),  # negative = improving (lower position)
        "content_age_days": g["content_age_days"].iloc[-1],
        "word_count": g["word_count"].iloc[-1],
        "volatility": g["avg_position"].std(),
        "days_observed": g["date"].nunique(),
    })

features = pd.DataFrame(rows)
features = features[features["days_observed"] >= 10].reset_index(drop=True)  # drop low-observation pages
print(features.shape)
features.describe()


## Step 3 — Scale features and check for leakage / degenerate columns

In [ ]:
# Step 3.1 — sanity checks before clustering
feature_cols = [
    "avg_position_mean", "ctr_mean", "impressions_sum", "clicks_sum",
    "impressions_trend", "ctr_trend", "position_trend",
    "content_age_days", "word_count", "volatility",
]

# leakage check: no feature should be a deterministic function of another
print(features[feature_cols].corr().round(2))

X = features[feature_cols].copy()
X["impressions_sum"] = np.log1p(X["impressions_sum"])
X["clicks_sum"] = np.log1p(X["clicks_sum"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## Step 4 — Choose k: elbow + silhouette

In [ ]:
# Step 4.1 — evaluate k = 3..9
inertias, sils = [], []
k_range = range(3, 10)

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_title("Elbow — inertia vs k")
axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia")

axes[1].plot(list(k_range), sils, marker="o", color="darkorange")
axes[1].set_title("Silhouette score vs k")
axes[1].set_xlabel("k"); axes[1].set_ylabel("silhouette")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/k_selection.png", dpi=150)
plt.show()

best_k = list(k_range)[int(np.argmax(sils))]
print("Suggested k (max silhouette):", best_k)


## Step 5 — Fit final clustering + baseline comparison
Baseline = simple quantile rule on `ctr_mean` and `impressions_trend` only
(a non-ML archetype split). The clustering result should beat this baseline
on silhouette / separation to justify the extra complexity.


In [ ]:
# Step 5.1 — final KMeans fit
K = best_k  # override manually here if you want to force a business-friendly k, e.g. 5 or 6
final_km = KMeans(n_clusters=K, n_init=25, random_state=42)
features["cluster"] = final_km.fit_predict(X_scaled)

sil_final = silhouette_score(X_scaled, features["cluster"])
print(f"Final k={K}, silhouette={sil_final:.3f}")


In [ ]:
# Step 5.2 — naive baseline: quantile split on ctr_mean x impressions_trend
def baseline_bucket(row):
    if row.ctr_mean >= features.ctr_mean.median() and row.impressions_trend >= 0:
        return "baseline_protect"
    if row.ctr_mean < features.ctr_mean.median() and row.impressions_trend >= 0:
        return "baseline_improve"
    if row.impressions_trend < 0:
        return "baseline_declining"
    return "baseline_monitor"

features["baseline_bucket"] = features.apply(baseline_bucket, axis=1)
print(features["baseline_bucket"].value_counts())
print("(cross-tab vs cluster to show clustering finds structure the baseline misses)")
pd.crosstab(features["cluster"], features["baseline_bucket"])


## Step 6 — Name archetypes from centroids and map to actions
Centroids are inspected on the **original (unscaled)** feature values so the
archetype names are human-readable, then each cluster is mapped to one of:
`protect / improve / rewrite / merge / prune / monitor`.


In [ ]:
# Step 6.1 — inspect centroids in original units
centroid_view = features.groupby("cluster")[feature_cols].mean().round(2)
centroid_view["n_pages"] = features["cluster"].value_counts().sort_index()
centroid_view


In [ ]:
# Step 6.2 — rule-based archetype naming from centroid position
# Edit the thresholds after looking at Step 6.1's actual centroid table for your data.
ARCHETYPE_RULES = {
    # cluster_id: (archetype_name, action, reason_code)
    # fill in after inspecting centroid_view, e.g.:
    # 0: ("Star performer", "protect", "high ctr, high impressions, flat/positive trend"),
    # 1: ("Sleeper", "improve", "low ctr, high impressions, stable position"),
    # 2: ("Declining", "rewrite", "negative impressions trend, worsening position"),
    # 3: ("Redundant", "merge", "low word count, overlapping topic, low unique traffic"),
    # 4: ("Dead weight", "prune", "near-zero impressions and clicks, old content"),
    # 5: ("Stable", "monitor", "flat trend across all metrics"),
}

features["archetype"] = features["cluster"].map(lambda c: ARCHETYPE_RULES.get(c, ("TBD", "monitor", "unmapped"))[0])
features["action"] = features["cluster"].map(lambda c: ARCHETYPE_RULES.get(c, ("TBD", "monitor", "unmapped"))[1])
features["reason_code"] = features["cluster"].map(lambda c: ARCHETYPE_RULES.get(c, ("TBD", "monitor", "unmapped"))[2])

features[["page_id", "cluster", "archetype", "action", "reason_code"]].head(10)


## Step 7 — Validation: stability across seeds and time split

In [ ]:
# Step 7.1 — stability check: re-run clustering with different seeds,
# measure how often pages stay in the same relative archetype (via Adjusted Rand Index)
from sklearn.metrics import adjusted_rand_score

seed_labels = []
for seed in [1, 7, 42, 99]:
    km_s = KMeans(n_clusters=K, n_init=10, random_state=seed)
    seed_labels.append(km_s.fit_predict(X_scaled))

ari_scores = []
for i in range(len(seed_labels)):
    for j in range(i + 1, len(seed_labels)):
        ari_scores.append(adjusted_rand_score(seed_labels[i], seed_labels[j]))

print("Mean pairwise ARI across seeds:", round(np.mean(ari_scores), 3))
print("(closer to 1.0 = more stable clustering; report this honestly in Limitations)")


In [ ]:
# Step 7.2 — time-aware split validation
# Refit on first half of the month only, check whether second-half behaviour
# of each page still falls in the same archetype the first half predicted.
half_mask = df_raw["date"] < df_raw["date"].quantile(0.5)
# (implementation left for your data's actual date column type —
#  rebuild the Step 2 feature table separately for first-half and second-half
#  data, refit KMeans on first-half only, and score how many pages'
#  second-half feature vector nearest-centroid still matches. Report the
#  agreement rate as a validation metric, not a guarantee.)


## Step 8 — Visualize archetypes (PCA projection)

In [ ]:
# Step 8.1 — 2D PCA projection colored by archetype
pca = PCA(n_components=2, random_state=42)
proj = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(proj[:, 0], proj[:, 1], c=features["cluster"], cmap="tab10", alpha=0.6, s=15)
ax.set_title("Content archetypes (PCA projection)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} var)")
plt.colorbar(scatter, label="cluster")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/archetype_pca.png", dpi=150)
plt.show()


## Step 9 — Export ranked action table

In [ ]:
# Step 9.1 — final public-safe export (page_id only, no URLs/queries/credentials)
export_cols = ["page_id", "archetype", "action", "reason_code",
               "ctr_mean", "impressions_trend", "position_trend", "content_age_days"]

# Rank within each action bucket by impressions_sum (biggest opportunity first)
features["_rank_key"] = features["impressions_sum"]
ranked = (
    features.sort_values(["action", "_rank_key"], ascending=[True, False])
    [export_cols]
    .reset_index(drop=True)
)

ranked.to_csv(f"{OUT_DIR}/ranked_content_actions.csv", index=False)
print(f"Exported {len(ranked)} rows to {OUT_DIR}/ranked_content_actions.csv")
ranked.head(15)


### Next steps for the capstone paper
1. Fill in `ARCHETYPE_RULES` in Step 6.2 once you see your real centroid table.
2. Copy the silhouette score, ARI stability score, and the two PNGs
   (`k_selection.png`, `archetype_pca.png`) into the **Results** section of
   `paper.md`.
3. Write the honest **Limitations** section using the Step 7 numbers —
   state clustering as *directional/decision-support*, not ground truth.
4. Commit this notebook into `work/` and update `submission/paper_url.txt`
   once the paper is deployed.
